# Virtual Packet Postprocessing Guide

This notebook demonstrates TARDIS's three spectrum generation methods with a focus on the new tracker-based virtual packet postprocessing workflow.

## Spectrum Generation Methods in TARDIS

TARDIS provides three complementary methods for generating synthetic spectra:

1. **Real Packet Spectrum**: Direct Monte Carlo output from escaping packets (noisy but unbiased)
2. **Formal Integral Spectrum**: Analytical post-processing using the converged radiation field
3. **Virtual Packet Spectrum**: Tracker-based variance reduction technique for smoother spectra

This guide focuses on virtual packets, which use a postprocessing architecture based on real packet tracking data.

See the [Virtual Packets Physics Documentation](../../physics_walkthrough/spectrum/virtualpackets.rst) for theoretical details.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from tardis import run_tardis
from tardis.io.atom_data import download_atom_data
from tardis.io.configuration.config_reader import Configuration

## Basic Usage

The workflow consists of two phases: transport simulation with tracking, then virtual packet generation.

In [ ]:
download_atom_data('kurucz_cd23_chianti_H_He_latest')

CONFIG_PATH = Path("../tardis_example.yml")
config = Configuration.from_yaml(CONFIG_PATH)

In [ ]:
sim = run_tardis(config)

Generate virtual packets from tracker data:

In [ ]:
sim.generate_virtual_spectrum()

Access all three spectrum types:

In [ ]:
spectrum_real = sim.spectrum_solver.spectrum_real_packets
spectrum_formal = sim.spectrum_solver.spectrum_integrated  
spectrum_virtual = sim.spectrum_solver.spectrum_virtual_packets

## Comparing the Three Methods

In [ ]:
%matplotlib inline
plt.figure(figsize=(12, 7))

spectrum_real.plot(label="Real packets (noisy)")
spectrum_formal.plot(label="Formal integral (smooth)")
spectrum_virtual.plot(label="Virtual packets (variance reduced)")

plt.xlim(3000 * u.AA, 9000 * u.AA)
plt.xlabel(r"Wavelength [$\AA$]")
plt.ylabel(r"Luminosity [$L_\odot / \AA$]")
plt.title("Comparison of TARDIS Spectrum Generation Methods")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Understanding the Tracker-Based Architecture

Virtual packets are generated from spawn events logged during Monte Carlo transport.

### Architecture Flow

```
MC Transport (with tracking enabled)
         ↓
tracker_full_df (spawn events logged)
         ↓
VirtualPacketSolver.generate_virtual_packets()
         ↓
VirtualPacketState (results)
         ↓
Spectrum generation
```

**Spawn events** include:
- Initial packet emission from photosphere
- Line interactions (resonant scattering)
- Electron scattering (ESCATTER) events

In [ ]:
tracker_df = sim.transport.transport_state.tracker_full_df
print(f"Total spawn events tracked: {len(tracker_df):,}")
print(f"Tracker columns: {list(tracker_df.columns)}")

## Accessing Virtual Packet Data

The `VirtualPacketState` object stores all virtual packet data with properties for backward compatibility.

In [ ]:
vp_state = sim.spectrum_solver.virtual_packet_state

print(f"Number of virtual packets: {len(vp_state.nus):,}")
print(f"Frequency range: {vp_state.nus.min():.2e} - {vp_state.nus.max():.2e} Hz")
print(f"Total energy: {vp_state.energies.sum():.2e} erg")

### Available Properties

The virtual packet state provides access to:
- `nus`: Packet frequencies
- `energies`: Packet energies  
- `initial_rs`: Initial radii
- `initial_mus`: Initial propagation angles
- `last_interaction_type`: Type of last interaction
- `last_interaction_in_nu`: Frequency at last interaction
- `last_interaction_in_r`: Radius of last interaction
- And more for visualization tools

In [ ]:
nus = vp_state.nus * u.Hz
energies = vp_state.energies * u.erg
initial_rs = vp_state.initial_rs * u.cm

wavelengths = nus.to(u.AA, equivalencies=u.spectral())

plt.figure(figsize=(10, 6))
plt.hist(wavelengths.value, bins=100, alpha=0.7, edgecolor='black')
plt.xlabel(r"Wavelength [$\AA$]")
plt.ylabel("Number of Virtual Packets")
plt.title("Virtual Packet Wavelength Distribution")
plt.grid(alpha=0.3)
plt.show()

## Performance: Tracker-Based Postprocessing

The new architecture uses numba parallelization for 10-30% performance improvement.

In [ ]:
import time

start = time.time()
sim.generate_virtual_spectrum()
elapsed = time.time() - start

print(f"Virtual packet generation time: {elapsed:.2f} seconds")
print(f"Packets generated: {len(vp_state.nus):,}")
print(f"Rate: {len(vp_state.nus)/elapsed:,.0f} packets/second")

### Benefits of Postprocessing

**Performance**: Numba-parallelized generation is faster than inline tracking

**Flexibility**: Generate multiple spectrum variations from a single simulation

**Memory efficiency**: Tracker is more compact than storing all virtual packet data

**Decoupling**: Spectrum generation can be optimized independently from transport

## Integration with Visualization Tools

Virtual packets work seamlessly with TARDIS visualization tools:

In [ ]:
from tardis.visualization import SDECPlotter

plotter = SDECPlotter.from_simulation(sim)
plotter.generate_plot_mpl(packet_wvl_range=[3000, 7000] * u.AA, nelements=5)

See the full [SDEC Plot Tutorial](../../analyzing_tardis/visualization/how_to_sdec_plot.ipynb) and [LIV Plot Tutorial](../../analyzing_tardis/visualization/how_to_liv_plot.ipynb) for more details.

## Troubleshooting

### Virtual packet state is None

**Problem**: `sim.spectrum_solver.virtual_packet_state` is `None`

**Solution**: Call `sim.generate_virtual_spectrum()` to generate virtual packets from tracker

### DeprecationWarning about virtual packets

**Problem**: Seeing warning when `no_of_virtual_packets > 0` in config

**Solution**: This is expected. Virtual packets now require explicit postprocessing call `sim.generate_virtual_spectrum()` after running simulation.

### Performance concerns

**Problem**: Is postprocessing slower than inline generation?

**Solution**: No! The new tracker-based approach with numba parallelization is 10-30% faster than the old inline method.

### Tracking not enabled

**Problem**: Tracker is empty or missing

**Solution**: Tracking is automatically enabled when `no_of_virtual_packets > 0`. If using older code, set `config.montecarlo.tracking.track_rpacket = True`.

## Summary

**Key Points:**
- TARDIS offers three spectrum methods: real packets, formal integral, virtual packets
- Virtual packets use tracker-based postprocessing for better performance  
- Call `sim.generate_virtual_spectrum()` after simulation to generate virtual packets
- Spawn events from `tracker_full_df` enable flexible spectrum generation
- Numba parallelization provides 10-30% speedup
- Full backward compatibility with visualization tools

For more details, see:
- [Virtual Packets Physics](../../physics_walkthrough/spectrum/virtualpackets.rst)
- [Formal Integral Method](../../physics_walkthrough/spectrum/formal_integral.rst) 
- [SDEC Plots](../../analyzing_tardis/visualization/how_to_sdec_plot.ipynb)
- [LIV Plots](../../analyzing_tardis/visualization/how_to_liv_plot.ipynb)